In [1]:
import pandas as pd
import numpy as np
from universe import UniverseBuilder
from backtester import BacktestEngine

# Quick test
ub = UniverseBuilder()

def momentum_metrics(df):

    r = df["return"]
    total_return = (1 + r).prod() - 1
    vol = r.std() * np.sqrt(12)

    years = len(df) / 12
    ann_return = (1 + total_return)**(1/years) - 1
    sharpe = ann_return / vol

    dd = df["nav"] / df["nav"].cummax() - 1

    return {
        "Total Return": total_return,
        "Annual Return": ann_return,
        "Vol": vol,
        "Sharpe": sharpe,
        "Max Drawdown": dd.min()
    }



In [2]:
builder = UniverseBuilder()
dates = builder.get_month_end_dates()

u = builder.get_universe(dates[0])

print("Universe size:", len(u))


Universe size: 991


In [3]:
print("Running Backtest...")
bt = BacktestEngine(ub)
results = bt.run()
if not results.empty:
    print("\nBacktest Results:")
    print(results.tail())
    metrics = bt.compute_metrics()
    print("\nMetrics:")
    for k, v in metrics.items():
        print(f"{k}: {v}")
else:
    print("No results found.")

Running Backtest...

Backtest Results:
          date    return  n_tickers  cumulative_return
245 2025-10-31 -0.005076       2492           2.946588
246 2025-11-28 -0.004240       2495           2.934094
247 2025-12-31  0.000167       2449           2.934583
248 2026-01-30  0.029483       2444           3.021103
249 2026-02-10  0.011946       2459           3.057193

Metrics:
Total Return: 205.72%
Annualized Return: 5.55%
Annualized Vol: 17.55%
Sharpe Ratio: 0.32
Max Drawdown: -53.74%
Count: 250


In [4]:
print(bt.results.columns)

Index(['date', 'return', 'n_tickers', 'cumulative_return'], dtype='str')


In [5]:
spy = bt.compute_spy_benchmark()

merged = bt.results.merge(spy, on="date", how="inner")

print(merged.tail())

          date    return  n_tickers  cumulative_return  spy_return   spy_nav
245 2025-10-31 -0.005076       2492           2.946588    0.023837  8.627028
246 2025-11-28 -0.004240       2495           2.934094    0.001950  8.643850
247 2025-12-31  0.000167       2449           2.934583    0.000797  8.650744
248 2026-01-30  0.029483       2444           3.021103    0.014738  8.778236
249 2026-02-10  0.011946       2459           3.057193    0.000217  8.780140


In [6]:
ub.compute_monthly_momentum()

print("Momentum column exists:",
      "momentum" in ub.prices.columns)

print(ub.prices["momentum"].describe())

print("Non-null momentum rows:",
      ub.prices["momentum"].notna().sum())


Momentum column exists: True
count    454368.000000
mean          0.082158
std           0.598085
min          -1.000000
25%          -0.198932
50%           0.013383
75%           0.234094
max           5.000000
Name: momentum, dtype: float64
Non-null momentum rows: 454368


In [7]:
mom_results = bt.run_momentum_strategy()

print(mom_results.tail())

print("Final NAV:", mom_results["nav"].iloc[-1])

          date    return  n_stocks       nav
236 2025-10-31  0.034772       235  6.357077
237 2025-11-28 -0.028322       234  6.177032
238 2025-12-31 -0.008086       229  6.127083
239 2026-01-30  0.074054       228  6.580820
240 2026-02-10  0.013286       229  6.668253
Final NAV: 6.668253137977465


In [8]:
momentum_metrics(mom_results)

{'Total Return': np.float64(5.668253137977465),
 'Annual Return': np.float64(0.0990808654371873),
 'Vol': np.float64(0.22025250300555244),
 'Sharpe': np.float64(0.4498512574664794),
 'Max Drawdown': np.float64(-0.5836863606592092)}

In [9]:
base = bt.run_momentum_base()
momentum_metrics(base)

{'Total Return': np.float64(4.731392214683698),
 'Annual Return': np.float64(0.09082652802637159),
 'Vol': np.float64(0.2309735649852942),
 'Sharpe': np.float64(0.3932334335842909),
 'Max Drawdown': np.float64(-0.7082614724747529)}

In [10]:
vol_scaled = bt.run_momentum_vol_scaled()
momentum_metrics(vol_scaled)

{'Total Return': np.float64(10.8871157547676),
 'Annual Return': np.float64(0.13117755623818406),
 'Vol': np.float64(0.24760219245272952),
 'Sharpe': np.float64(0.5297915779288891),
 'Max Drawdown': np.float64(-0.6157071998912592)}

In [11]:
overlap = bt.apply_overlapping(vol_scaled)
momentum_metrics(overlap)

{'Total Return': np.float64(10.8871157547676),
 'Annual Return': np.float64(0.13117755623818406),
 'Vol': np.float64(0.24760219245272952),
 'Sharpe': np.float64(0.5297915779288891),
 'Max Drawdown': np.float64(-0.3466125438202552)}

In [12]:
beta_adj, beta = bt.beta_neutralize(overlap, spy)
print(beta)
momentum_metrics(beta_adj)

-0.20937238871991434


{'Total Return': np.float64(10.8871157547676),
 'Annual Return': np.float64(0.13117755623818406),
 'Vol': np.float64(0.24760219245272952),
 'Sharpe': np.float64(0.5297915779288891),
 'Max Drawdown': np.float64(-0.5593337567128345)}